In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys

project_root = '/content/drive/MyDrive/GNNs/HIV inhibitors-GNN'
os.makedirs(project_root, exist_ok=True)
os.chdir(project_root)
sys.path.append(project_root)

In [3]:
!pip install torch==2.4.0

import torch
torch_version = torch.__version__
print("PyTorch version:", torch_version)

!pip install rdkit

!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-${torch_version}.html

!pip install torch_geometric


PyTorch version: 2.4.0+cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 47.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-.4.0+cu121.html
ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.8 MB/s eta 0:00:00


In [4]:

# !pip install -q mlflow pyngrok

# import os
# from getpass import getpass
# import mlflow
# from pyngrok import ngrok
# import time
# import subprocess

# BASE_DIR = "/content/drive/MyDrive/GNNs/HIV inhibitors-GNN"

# os.makedirs(f"{BASE_DIR}/outputs/mlruns", exist_ok=True)
# os.makedirs(f"{BASE_DIR}/outputs/artifacts", exist_ok=True)

# MLFLOW_TRACKING_URI = f"sqlite:///{BASE_DIR}/outputs/mlruns/mlflow.db"
# mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# EXPERIMENT_NAME = "HIV_GNN_Classification"

# artifact_location = f"file://{BASE_DIR}/outputs/artifacts"

# try:
#     experiment_id = mlflow.create_experiment(
#         EXPERIMENT_NAME,
#         artifact_location=artifact_location
#     )
#     print("Created experiment:", experiment_id)
# except:
#     experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
#     experiment_id = experiment.experiment_id

# mlflow.set_experiment(EXPERIMENT_NAME)

# print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
# print(f"Experiment: {EXPERIMENT_NAME}")

# print("\nngrok authtoken............")
# ngrok_token = getpass("Paste your ngrok authtoken here and press Enter: ")

# if ngrok_token.strip():
#     !ngrok config add-authtoken {ngrok_token}
#     print("ngrok authtoken saved!")
# else:
#     print("No token entered. Public UI will not work.")

# # === Kill old processes (clean start) ===
# ngrok.kill()
# !pkill -f "mlflow ui" || true
# time.sleep(2)

# print("\nStarting MLflow UI...")

# # ✅ FIXED + IMPROVED command
# subprocess.Popen([
#     "mlflow", "ui",
#     "--host", "0.0.0.0",
#     "--port", "5000",
#     "--workers", "1",
#     "--allowed-hosts", "*",
#     "--backend-store-uri", MLFLOW_TRACKING_URI,
#     "--default-artifact-root", f"file://{BASE_DIR}/outputs/artifacts"
# ])

# # Wait until the port is actually listening
# print("Waiting for MLflow to bind to port 5000...")
# for i in range(15):
#     time.sleep(1)
#     import socket
#     sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#     result = sock.connect_ex(('127.0.0.1', 5000))
#     sock.close()
#     if result == 0:
#         print(f"✅ MLflow UI is ready on port 5000 (took {i+1}s)")
#         break
#     if i == 14:
#         print("⚠️ MLflow did not start in time. Check the next lines.")

# time.sleep(2)

# # Start ngrok
# try:
#     public_url = ngrok.connect(5000, "http").public_url
#     print(f"\n🎉 MLflow UI is live at: {public_url}")
#     print("   → Open this link in a new tab")
#     print("   → Keep this Colab cell running (do not close the tab)")
# except Exception as e:
#     print(f"\n❌ Could not create ngrok tunnel: {e}")

# # === Diagnostic (very useful) ===
# print("\n=== Running processes ===")
# !ps aux | grep mlflow
# print("\n=== Port 5000 status ===")
# !lsof -i :5000


# Install MLflow and Ngrok
!pip install -q mlflow pyngrok

import os
import mlflow
from pyngrok import ngrok
import subprocess
import socket
import time
from getpass import getpass

# ---------------------------
# Directories
# ---------------------------
BASE_DIR = "/content/drive/MyDrive/GNNs/HIV inhibitors-GNN"
MLRUNS_DIR = f"{BASE_DIR}/outputs/mlruns"
ARTIFACT_DIR = f"{BASE_DIR}/outputs/artifacts"

os.makedirs(MLRUNS_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---------------------------
# MLflow tracking URI
# ---------------------------
MLFLOW_TRACKING_URI = f"sqlite:///{MLRUNS_DIR}/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("Tracking URI:", mlflow.get_tracking_uri())

# ---------------------------
# Experiment setup
# ---------------------------
EXPERIMENT_NAME = "HIV_GNN_Classification"
artifact_location = f"file://{ARTIFACT_DIR}"

# Try to get existing experiment first
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifact_location)
    print(f"Created experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

mlflow.set_experiment(EXPERIMENT_NAME)

# ---------------------------
# Ngrok setup
# ---------------------------
token = getpass("Paste your ngrok authtoken: ")
if token.strip():
    !ngrok config add-authtoken {token}

# Clean old processes
ngrok.kill()
!pkill -f mlflow || true
time.sleep(2)

# ---------------------------
# Start MLflow server
# ---------------------------
print("Starting MLflow server...")
subprocess.Popen([
    "mlflow", "server",
    "--host", "0.0.0.0",
    "--port", "5000",
    "--backend-store-uri", MLFLOW_TRACKING_URI,
    "--default-artifact-root", f"file://{ARTIFACT_DIR}",
    "--workers", "1",
    "--allowed-hosts", "*",
    "--cors-allowed-origins", "*"
])

# Wait for server
for i in range(20):
    time.sleep(1)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    if sock.connect_ex(("127.0.0.1", 5000)) == 0:
        print("✅ MLflow server ready")
        break
    sock.close()

# ---------------------------
# Start ngrok tunnel
# ---------------------------
public_url = ngrok.connect(5000).public_url
print("\n🎉 MLflow UI:")
print(public_url)
print("\nKeep this Colab cell running!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.0/814.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 9.5 MB/s eta 0:00:00
Tracking URI: sqlite:////content/drive/MyDrive/GNNs/HIV inhibitors-GNN/outputs/mlruns/mlflow.db
Using existing experiment: HIV_

In [5]:
%run src/training/train.py --epochs 100 --use_pos_weight 1 --use_weighted_sampler 0

Using pos_weight: True, Using weighted sampler: False
loading datasets.........

Derived parameters......

Get loaders..................
train_set_size:32568 | val_set_size:4440
Class Counts: [31421  1147]
Class distribution: 31421 negatives(0), 1147 positives(1)
pos_weight = 5.2

Create Model.................

Create loss function and Optimizer.................

Load Checkpoint.................
no checkpoint found. starting from epoch 0.
Starting new MLflow run: 6aa5103efb864d3897969b768dc2243f

Log Params in MLflow.........................



Start training loop.....................




Training......:   0%|          | 0/509 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
Training......: 100%|██████████| 509/509 [00:14<00:00, 35.47it/s]


Type:train| Epoch:0 | AP: 0.0358 | AUC: 0.4811 | Recall@0.1: 0.7515257192676548 | Precision@0.1: 0.033096563639854096 | Recall@0.2: 0.3225806451612903 | Precision@0.2: 0.03425925925925926 | Recall@0.3: 0.13426329555361813 | Precision@0.3: 0.040451799317047545 | Recall@0.5: 0.043591979075850044 | Precision@0.5: 0.04222972972972973 

Epoch: 0 | Train loss: 0.6742932568696581




Training......:   0%|          | 0/509 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
Training......: 100%|██████████| 509/509 [00:13<00:00, 38.31it/s]


Type:train| Epoch:1 | AP: 0.0413 | AUC: 0.5284 | Recall@0.1: 0.909328683522232 | Precision@0.1: 0.034577642222516906 | Recall@0.2: 0.2153443766346992 | Precision@0.2: 0.047254639372489 | Recall@0.3: 0.05056669572798605 | Precision@0.3: 0.05375347544022243 | Recall@0.5: 0.010462074978204011 | Precision@0.5: 0.05240174672489083 

Epoch: 1 | Train loss: 0.5169964748894779




...

Final Results =>  Type:Validation-loss model| AP: 0.0577 | AUC: 0.6038 | Final Recall@0.1: 0.19736842105263158 | Final Precision@0.1: 0.08571428571428572 | Final Recall@0.2: 0.0 | Final Precision@0.2: 0.0 | Final Recall@0.3: 0.0 | Final Precision@0.3: 0.0 | Final Recall@0.5: 0.0 | Final Precision@0.5: 0.0 



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Final Results =>  Type:Test-loss model| AP: 0.0655 | AUC: 0.6166 | Final Recall@0.1: 0.24305555555555555 | Final Precision@0.1: 0.11217948717948718 | Final Recall@0.2: 0.0 | Final Precision@0.2: 0.0 | Final Recall@0.3: 0.0 | Final Precision@0.3: 0.0 | Final Recall@0.5: 0.0 | Final Precision@0.5: 0.0 



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

<Figure size 640x480 with 0 Axes>